In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image
import slideio

from joblib import Parallel, delayed
from joblib import parallel_config

# from skimage.transform import resize
# from skimage import data
from skimage import color
from skimage import morphology
# from skimage import segmentation

import os
import gc
import time
import os

In [2]:
cpu_count = os.cpu_count()
cpu_count

24

In [3]:
os.listdir()

['.ipynb_checkpoints',
 '17899.svs',
 '18136.svs',
 'CreateTiles_connectedblocks.ipynb',
 'CreateTiles_OpenSlide.ipynb',
 'CreateTiles_singleblock.ipynb',
 'SVS_Tiling.ipynb',
 'tiles',
 'Tiles_17899.csv',
 'Tiles_18136.csv',
 'VerifyEfficientNet']

In [4]:
#Clear tile directory
dstdir = 'tiles'
tilefiles = os.listdir(dstdir)
for tf in tilefiles:
    os.remove(os.path.join(dstdir, tf))

In [5]:
#list to process
svsfiles = [wsi for wsi in os.listdir() if '.svs' in wsi]
svsfiles

['17899.svs', '18136.svs']

In [6]:
#tilesize for processsing
#1024x1024 crops resized to 224x224

ts = 1024
CNNsize = 224

#Allowable pct of saturated area - area where R+G+B > threshold, white ares of slide, no tissue present
#Keep tiles with at least 1-pct having tissue: 1-.85 --> 15% of tile needs to have tissue present
###########################################
pct = 0.5
###########################################

In [7]:
#Processes tile for tissue presence
def tissue_present(patch, pct):
    imgarray = np.array(patch)
    imgsum = np.sum(imgarray, axis=2)

    imgarea = imgarray.shape[0] * imgarray.shape[1]

    saturated = np.where(imgsum > 700)
    #Reject if less than pct% has tissue
    if saturated[0].shape[0] / imgarea > pct:
        #Not enough tissue
        return False
    else:
        #Enough tissue
        return True

In [8]:
def parallel_patch(coords, boxnames, count):
    #Extract patch
    if count % 50 == 0:
        print('Processing patch ', count)

    #Extract tile
    x1, y1, x2, y2 = coords
    box = scene.read_block((x1, y1, x2-x1, y2-y1), size=(int((x2-x1)*CNNsize/ts), 0))
    
    #Divide box into tiles
    for i, tilename in enumerate(boxnames):
        tile = box[:, i*CNNsize:i*CNNsize+CNNsize]
        
        #to PIL image to resize, save
        tile = Image.fromarray(np.uint8(tile))
        tile.save(os.path.join('tiles', tilename))
    return True

In [9]:
svsfiles

['17899.svs', '18136.svs']

In [10]:
#Iterate over SVS, saving tiles and Summaries
start = time.time()
for svsfile in svsfiles:

    fname = svsfile.split('.')[0]
    print('Processing file ', fname)

    #repository for tile information to write to csv
    sourcefiles = []
    tilenames = []
    coordinates = []

    #Open SVS
    slide = slideio.open_slide(svsfile,'SVS')
    scene = slide.get_scene(0)

    #width, height
    svs_size = scene.size
    print(svs_size)

    #################################################################
    #Create mask 1/64 of img size: 1024x1024 represented by 16x16
    #svsimage and imgmask used to check for presence of tissue
    scale = 64
    svsimage = scene.read_block((0,0,svs_size[0], svs_size[1]), size=(svs_size[0]//scale,0))

    # Compute a mask
    lum = color.rgb2gray(svsimage)
    imgmask = morphology.remove_small_holes(morphology.remove_small_objects(lum < 0.9, 500), 500)
    imgmask = morphology.opening(imgmask, morphology.disk(3))
    masksize = imgmask.shape
    mask_ts = ts // scale
    print('Mask created')

    # y steps, x steps
    steps = [masksize[0]//mask_ts, masksize[1]//mask_ts] 

    #Unique name
    patchcount = 0
    
    #Test tile sites
    for i in range(steps[0]): #ysteps
        for j in range(steps[1]):   #xsteps
            #Mask test
            masktest = imgmask[mask_ts*i:mask_ts*i+mask_ts, mask_ts*j:mask_ts*j+mask_ts]
            if not masktest.any():
                continue
                
            #Tissue test
            tissuetest = svsimage[mask_ts*i:mask_ts*i+mask_ts, mask_ts*j:mask_ts*j+mask_ts]
            if not tissue_present(tissuetest, pct):
                continue

            #Add to list for extraction for parallel setup if both checks passed
            tilename = fname + '_' + str(patchcount) + '.png'
            
            #Coordinate order: x1, y1, x2, y2
            coordinates.append([ts*j, ts*i, ts*j+ts, ts*i+ts])
            tilenames.append(tilename)

            patchcount += 1

    #Analyze for connected tiles (boxes) in a row - have SlideIO extract entire Block at once if connected (much faster)
    boxes = []
    boxnames = []
    names = []
    for i, coords in enumerate(coordinates):  
        #Initialize w/ first tile
        if i == 0: 
            x1_box, y1_box, x2_box, y2_box = coords
            names.append(fname + '_' + str(i) + '.png')
            continue
            
        #New coordinates for test
        x1_new, y1_new, x2_new, y2_new = coords
        

        #Same row, connected
        if x1_new == x2_box and y1_new == y1_box:
            #y1_box, y2_box unchanged - same row
            #update right edge only
            x2_box = x2_new
        else:
            #write existing box, reset counter, start new box
            boxes.append([x1_box, y1_box, x2_box, y2_box])
            boxnames.append(names)
            names = []
            x1_box, y1_box, x2_box, y2_box = coords
            
        names.append(fname + '_' + str(i) + '.png')

        
    #Process tiles in parallel
    print('Extracting ', len(coordinates), 'tiles')
    with parallel_config(backend='threading', n_jobs=12):
        results = Parallel()(delayed(parallel_patch)(coords, boxnames[i], i) for i, coords in enumerate(boxes))
        # results = Parallel()(delayed(parallel_patch)(coords, tilenames[i], i) for i, coords in enumerate(coordinates))

    
    #################################################################

    ################################################################
    #Write info to csv
    info = {}
    sourcefiles.append(svsfile)
    info['sourcefiles'] = sourcefiles * len(boxes)
    info['coords(x1, y1, x2, y2)'] = boxes
    info['tilenames'] = boxnames

    #Dictionary to dataframe, save to csv
    df_info = pd.DataFrame(info)
    summaryfile = 'Tiles_' + fname + '.csv'
    df_info.to_csv(summaryfile, index=True)

    ######################################################################
    gc.collect()

finish = time.time()
print('Elapsed, ', finish - start)

Processing file  17899
(123504, 81232)
Mask created
Extracting  4320 tiles
Processing patch  0
Processing patch  50
Processing patch  100
Processing patch  150
Processing file  18136
(137448, 72678)
Mask created
Extracting  3387 tiles
Processing patch  0
Processing patch  50
Processing patch  100
Processing patch  150
Processing patch  200
Processing patch  250
Elapsed,  13.247735977172852
